## Agents


## Agent = Tools + Harness
> **An agent is a model calling tools in a loop until a given task is complete.**
> **A harness is everything around that loop.**

**The model** is the raw language model — GPT, Claude, Gemini, whatever you plug in. 
**The harness** is everything that turns that raw capability into something useful:
- The **system prompt** — instructions on how the agent should behave
- The **tools** — what it's actually allowed to reach for and use
- The **middleware** — checkpoints that shape its behavior at every step


### Forgetting Issue with Agents

<img src="../../assets/agentic_loop.png" width="500" height="400">
<img src="../../assets/core_agent_loop.svg" width="600" height="400">
<img src="../../assets/agent_model_harness.svg" width="600" height="400">


In [23]:
import os
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents.structured_output import ToolStrategy
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
import sqlite3
import requests

import sys
sys.path.append('..')
from utils.helper import pretty_print_messages

In [6]:
if not os.getenv("TAVILY"):
    raise ValueError("TAVILY environment variable is not set.")
TAVILY = os.getenv("TAVILY")

In [7]:
tavily_search = TavilySearch(max_results=2, topic="general", tavily_api_key=TAVILY)

In [8]:
def setup_database():
  conn = sqlite3.connect("tripmate.db")
  cur = conn.cursor()
  cur.execute("""
    CREATE TABLE IF NOT EXISTS trips (
        trip_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        destination TEXT,
        start_date TEXT,
        end_date TEXT,
        status TEXT DEFAULT 'confirmed'
    )
""")
  conn.commit()
  conn.close()
setup_database()

In [11]:
@tool
def save_trip(user_id: str, destination: str, start_date: str, end_date: str) -> str:
    """Save a new trip to the real database."""
    conn = sqlite3.connect("tripmate.db")
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO trips (user_id, destination, start_date, end_date) VALUES (?, ?, ?, ?)",
        (user_id, destination, start_date, end_date),
    )
    conn.commit()
    trip_id = cur.lastrowid
    conn.close()
    return f"Trip #{trip_id} saved: {destination}, {start_date} to {end_date}."

@tool
def get_saved_trips(user_id: str) -> str:
    """Look up all saved trips for a user from the real database."""
    conn = sqlite3.connect("tripmate.db")
    cur = conn.cursor()
    cur.execute("SELECT trip_id, destination, start_date, end_date, status FROM trips WHERE user_id = ?", (user_id,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return "No saved trips found for this user."
    return "\n".join(f"Trip #{r[0]}: {r[1]} ({r[2]} to {r[3]}) -- {r[4]}" for r in rows)


In [12]:
# Prove this is REAL persistence -- save, then read back
print(save_trip.invoke({"user_id": "rohan_01", "destination": "Bali", "start_date": "2026-09-01", "end_date": "2026-09-10"}))
print(get_saved_trips.invoke({"user_id": "rohan_01"}))

Trip #1 saved: Bali, 2026-09-01 to 2026-09-10.
Trip #1: Bali (2026-09-01 to 2026-09-10) -- confirmed


In [13]:
@tool
def get_real_weather(city: str) -> str:
    """Get the REAL current weather for a city, using a live weather API."""
    # Step 1: geocode the city name into latitude/longitude
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    )
    geo_data = geo_response.json()
    if not geo_data.get("results"):
        return f"Could not find a location matching '{city}'."

    location = geo_data["results"][0]
    lat, lon = location["latitude"], location["longitude"]

    # Step 2: fetch the real current weather at that exact location
    weather_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": "true"},
    )
    weather_data = weather_response.json()
    current = weather_data.get("current_weather", {})

    return (
        f"Real current weather in {location['name']}, {location.get('country', '')}: "
        f"{current.get('temperature')}°C, wind {current.get('windspeed')} km/h."
    )

print("Tool defined -- run the cell below to hit the LIVE API for real.")


Tool defined -- run the cell below to hit the LIVE API for real.


In [16]:

class NewTripRequest(BaseModel):
    """A request to plan a new trip."""
    user_id: str
    destination: str
    start_date: str = Field(description="Format: YYYY-MM-DD")
    end_date: str = Field(description="Format: YYYY-MM-DD")

class ModifyTripRequest(BaseModel):
    """A request to change an existing trip."""
    user_id: str
    trip_id: int
    change_description: str

class CancelTripRequest(BaseModel):
    """A request to cancel an existing trip."""
    user_id: str
    trip_id: int


### Long Term Memory via InMemoryStore

In [20]:
travel_store = InMemoryStore()

In [ ]:
@tool
def save_travel_style(user_id: str, style: str, runtime: ToolRuntime) -> str:
    """Save a traveler's preferred trip style (e.g. budget, luxury, adventure) for future visits."""
    runtime.store.put((user_id, "preferences"), "travel_style", {"value": style})
    return f"Noted -- I'll remember you prefer {style} travel."

@tool
def recall_travel_style(user_id: str, runtime: ToolRuntime) -> str:
    """Recall a traveler's preferred trip style, if saved before."""
    result = runtime.store.get((user_id, "preferences"), "travel_style")
    return result.value["value"] if result else "No travel style saved yet for this user."


In [21]:

print("Tools defined -- runtime.store is a genuinely separate memory system from the")
print("SQLite database above. Trips are structured records; preferences are lightweight facts.")


Tools defined -- runtime.store is a genuinely separate memory system from the
SQLite database above. Trips are structured records; preferences are lightweight facts.


### Dynamic Tool Gating

In [ ]:
@tool
def book_premium_concierge(destination: str) -> str:
    """Book a dedicated human concierge for trip planning. Premium members only."""
    return f"Premium concierge assigned for your {destination} trip."


In [24]:

@wrap_model_call
def gate_premium_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose book_premium_concierge to premium members."""
    is_premium = request.state.get("is_premium_member", False)
    if not is_premium:
        allowed = [t for t in request.tools if t.name != "book_premium_concierge"]
        request = request.override(tools=allowed)
    return handler(request)


In [25]:
class TravelerContext(BaseModel):
    user_id: str
    home_currency: str
    membership_tier: str
    is_premium_member: bool

In [ ]:
tripmate_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[
        tavily_search,
        get_real_weather,
        save_trip,
        get_saved_trips,
        save_travel_style,
        recall_travel_style,
        book_premium_concierge,
    ],
    system_prompt=(
        "You are TripMate, a real travel planning assistant. Check real weather before "
        "recommending destinations. Save trips when confirmed. Remember travel style preferences"
        "Please make sure that you give the answer after running for 10  times and don't keep on running, be quick"
    ),
    middleware=[gate_premium_tools],
    checkpointer=InMemorySaver(),
    store=travel_store,
    context_schema=TravelerContext,
    name="tripmate_agent",
    response_format=ToolStrategy(Union[NewTripRequest, ModifyTripRequest, CancelTripRequest])

)

In [ ]:
config = {"configurable": {"thread_id": 'Rohan Trip Request'}}
result = tripmate_agent.invoke({"messages":[('user','I am rohan_01, plan a trip to Bali from 2026-09-01 to 2026-09-10 ')]},config=config)
print(result['structured_response'])

In [ ]:
pretty_print_messages(result)

In [ ]:
config = {"configurable": {"thread_id": "rohan-planning-session"}}
